# ISLES'24: Dual-Encoder Multi-Task Stroke Segmentation

**Mô tả:** Pipeline huấn luyện sử dụng Dual-Encoder (ResNet50 + DenseNet121) với trọng số RadImageNet trên 2 GPU T4.

In [ ]:
# Cài đặt các thư viện bổ trợ (nếu thiếu)
!pip install -q PyYAML matplotlib tqdm

## 1. Cài đặt Thư viện và Đồng bộ mã nguồn
Cell này đảm bảo môi trường có đủ các thư viện cần thiết và tải code mới nhất từ GitHub.

In [ ]:
import os
import sys

REPO_URL="https://github.com/tanminh51nbn/isles24-Stroke-multi-task-seg.git"
REPO_DIR="/kaggle/working/isles24-Stroke-multi-task-seg"
BRANCH="main" # Hoặc branch bạn đang sử dụng

if not os.path.exists(REPO_DIR):
    print("--- Cloning repository ---")
    !git clone --quiet {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !git checkout --quiet {BRANCH}
else:
    print("--- Repository exists, updating code ---")
    os.chdir(REPO_DIR)
    !git fetch --quiet origin
    !git checkout --quiet {BRANCH}
    !git pull origin {BRANCH} --quiet

# Quan trọng: Thêm thư mục src vào sys.path để Notebook nhận diện được các module
SRC_PATH = os.path.join(REPO_DIR, "src")
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

os.chdir(SRC_PATH)
print(f"Working Directory: {os.getcwd()}")

## 2. Cấu hình đường dẫn Tùy chỉnh

In [ ]:
# ─── PATHS ───
DATASET_DIR = "/kaggle/input/isles24-npy-dataset/ISLES24_NPY_Dataset"
CTA_WEIGHTS = "/kaggle/input/radimagenet-pytorch/ResNet50.pt"
PERF_WEIGHTS = "/kaggle/input/radimagenet-pytorch/DenseNet121.pt"
OUTPUT_DIR = "/kaggle/working/outputs"
METADATA_PATH = "/kaggle/working/dataset_metadata.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Khởi tạo Metadata (Cho Smart Sampling) ───
from data.metadata_builder import scan_dataset

if not os.path.exists(METADATA_PATH):
    print("--- Đang quét dataset để tạo metadata... ---")
    scan_dataset(DATASET_DIR, METADATA_PATH)
else:
    print(f"Metadata đã tồn tại tại: {METADATA_PATH}")

## 3. Thực thi Huấn luyện (DDP Mode)

In [ ]:
!python PINELINE.py \
    --dataset_dir {DATASET_DIR} \
    --output_dir {OUTPUT_DIR} \
    --cta_weights {CTA_WEIGHTS} \
    --perf_weights {PERF_WEIGHTS} \
    --metadata_path {METADATA_PATH}

## 4. Kết quả

In [ ]:
from IPython.display import Image, display

curve_path = os.path.join(OUTPUT_DIR, "training_curves.png")
if os.path.exists(curve_path):
    display(Image(filename=curve_path))
else:
    print("Chưa có file kết quả huấn luyện.")